# 6-5 Autograd 디버깅과 안전한 평가 — 기본

직접 작성한 코드와 저장된 실행 결과를 정리했습니다.


In [1]:
import random
import json
import math
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset, random_split

# 실습 결과가 매번 비슷하게 나오도록 seed를 고정합니다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


In [2]:
def validate(model, x, y, loss_fn):
    # TODO: 평가 모드로 전환하세요.
    model.eval()
    # TODO: no_grad 안에서 계산하세요.
    with torch.no_grad():
      pred = model(x)
      loss = loss_fn(pred, y)
      return float(loss.detach())

model = nn.Linear(3, 1)
x = torch.randn(10, 3)
y = torch.randn(10, 1)
print('valid loss:', validate(model, x, y, nn.MSELoss()))

valid loss: 1.2523126602172852


In [3]:
logits = torch.tensor([[2.0, 0.1], [0.2, 1.3], [1.0, 0.5]], requires_grad=True)
y = torch.tensor([0, 1, 1])

# TODO: metric 계산용 logits를 detach하세요.
metric_logits = logits.detach()
# TODO: 예측 label을 계산하세요.
pred = metric_logits.argmax(dim=1)
# TODO: accuracy를 계산하세요.
acc = (pred==y).float().mean()
print('pred:', pred.tolist())
print('accuracy:', float(acc))

pred: [0, 1, 0]
accuracy: 0.6666666865348816


In [6]:
model = nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
x = torch.randn(5, 2)
y = torch.randn(5, 1)
loss_fn = nn.MSELoss()

pred = model(x)
loss = loss_fn(pred, y)
optimizer.zero_grad()
loss.backward()
optimizer.step()

# TODO: 같은 loss를 재사용하지 말고, pred와 loss를 다시 계산하세요.
optimizer.zero_grad()
pred = model(x)
loss = loss_fn(pred, y)
loss.backward()
optimizer.step()
print('done')

done


In [5]:
model = nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
x = torch.randn(5, 2)
y = torch.randn(5, 1)
loss_fn = nn.MSELoss()

pred = model(x)
loss = loss_fn(pred, y)
optimizer.zero_grad()
loss.backward()
optimizer.step()

pred = model(x)
loss = loss_fn(pred, y)
optimizer.zero_grad()
loss.backward()
optimizer.step()
print('done')

done
